In [1]:
import os
import numpy as np
import pandas as pd

print("Baseline modeling environment ready")

Baseline modeling environment ready


In [2]:
ml_file = "../data/processed/features/ml_dataset.csv"

ml_dataset = pd.read_csv(ml_file)

print("Shape:", ml_dataset.shape)
print("Patients:", ml_dataset["patient_id"].nunique())

Shape: (11109, 17)
Patients: 44


In [3]:
print(ml_dataset.columns.tolist())

['patient_id', 'mean_rr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf', 'window_id', 'rr_count', 'sdnn_trend', 'rmssd_trend', 'pnn50_trend', 'age', 'sex', 'nyha_class', 'risk_label']


In [4]:
feature_columns = [
    "mean_rr",
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_power",
    "hf_power",
    "lf_hf",
    "rr_count",
    "sdnn_trend",
    "rmssd_trend",
    "pnn50_trend"
]

X = ml_dataset[feature_columns]
y = ml_dataset["risk_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11109, 11)
y shape: (11109,)


In [5]:
from sklearn.model_selection import train_test_split

patient_labels = (
    ml_dataset[
        ["patient_id", "risk_label"]
    ]
    .drop_duplicates()
)

print(patient_labels.shape)
print(patient_labels["risk_label"].value_counts())

(44, 2)
risk_label
1    32
0    12
Name: count, dtype: int64


In [6]:
train_patients, test_patients = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=42,
    stratify=patient_labels["risk_label"]
)

print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))

Training patients: 35
Testing patients: 9


In [7]:
print("Training class distribution:")
print(train_patients["risk_label"].value_counts())

print("\nTesting class distribution:")
print(test_patients["risk_label"].value_counts())

Training class distribution:
risk_label
1    25
0    10
Name: count, dtype: int64

Testing class distribution:
risk_label
1    7
0    2
Name: count, dtype: int64


In [8]:
overlap = set(train_patients["patient_id"]) & set(
    test_patients["patient_id"]
)

print("Patient overlap:", overlap)

Patient overlap: set()


In [9]:
train_ids = set(train_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

train_df = ml_dataset[
    ml_dataset["patient_id"].isin(train_ids)
].copy()

test_df = ml_dataset[
    ml_dataset["patient_id"].isin(test_ids)
].copy()

print("Training windows:", len(train_df))
print("Testing windows:", len(test_df))

Training windows: 8940
Testing windows: 2169


In [10]:
print(
    "Training patients:",
    train_df["patient_id"].nunique()
)

print(
    "Testing patients:",
    test_df["patient_id"].nunique()
)

Training patients: 35
Testing patients: 9


In [11]:
X_train = train_df[feature_columns]
y_train = train_df["risk_label"]

X_test = test_df[feature_columns]
y_test = test_df["risk_label"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8940, 11)
X_test: (2169, 11)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


In [13]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [14]:
y_pred = logistic_model.predict(X_test_scaled)

y_prob = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

print("Predictions generated")

Predictions generated


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Results")
print("---------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUROC    :", auc)

Logistic Regression Results
---------------------------
Accuracy : 0.5850622406639004
Precision: 0.7424522489217499
Recall   : 0.7142857142857143
F1 Score : 0.7280966767371602
AUROC    : 0.4144187796845292


In [16]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[  64  418]
 [ 482 1205]]


In [17]:
print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))

print("Training class distribution:")
print(train_patients["risk_label"].value_counts())

print("Testing class distribution:")
print(test_patients["risk_label"].value_counts())

print("Patient overlap:", overlap)

Training patients: 35
Testing patients: 9
Training class distribution:
risk_label
1    25
0    10
Name: count, dtype: int64
Testing class distribution:
risk_label
1    7
0    2
Name: count, dtype: int64
Patient overlap: set()


In [18]:
print(
    test_df[
        ["patient_id", "risk_label"]
    ].drop_duplicates().sort_values("patient_id")
)

      patient_id  risk_label
240        chf02           1
470        chf03           1
1665       chf08           1
4860      chf206           1
5111      chf207           1
5626      chf209           1
6890      chf214           0
7630      chf217           0
10563     chf228           1


In [19]:
print(
    test_df.groupby(
        ["patient_id", "risk_label"]
    ).size()
)

patient_id  risk_label
chf02       1             230
chf03       1             240
chf08       1             240
chf206      1             251
chf207      1             227
chf209      1             226
chf214      0             203
chf217      0             279
chf228      1             273
dtype: int64


In [21]:
test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

test_results["predicted_probability"] = y_prob
test_results["predicted_class"] = y_pred

In [38]:
patient_results = (
    test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
       #predicted_class=("predicted_class", "mean"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [39]:
patient_results["predicted_class"] = (
    patient_results["mean_probability"] >= 0.5
).astype(int)

In [34]:
print(patient_results)

  patient_id  risk_label  mean_probability  median_probability  \
0      chf02           1          1.000000            1.000000   
1      chf03           1          0.699295            0.713641   
2      chf08           1          0.491855            0.508818   
3     chf206           1          0.616369            0.637882   
4     chf207           1          0.997221            0.997793   
5     chf209           1          0.381264            0.384402   
6     chf214           0          0.986767            0.987077   
7     chf217           0          0.639977            0.633862   
8     chf228           1          0.495592            0.507719   

   predicted_class  window_count  
0                1           230  
1                1           240  
2                1           240  
3                1           251  
4                1           227  
5                0           226  
6                1           203  
7                1           279  
8                1      

In [40]:
patient_y_true = patient_results["risk_label"]

#patient_y_pred = patient_results["predicted_class"]

patient_y_prob = patient_results["mean_probability"]

In [42]:
print("\nPatient-level confusion matrix:")
print(
    confusion_matrix(
        patient_y_true,
        patient_y_pred
    )
)


Patient-level confusion matrix:
[[0 2]
 [1 6]]


In [45]:
test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

test_results["predicted_probability"] = y_prob
#test_results["predicted_class"] = y_pred

In [44]:
patient_y_true = patient_results["risk_label"]
patient_y_pred = patient_results["predicted_class"]
patient_y_prob = patient_results["mean_probability"]

In [46]:
print("Patient-level Accuracy:",
      accuracy_score(patient_y_true, patient_y_pred))

print("Patient-level Precision:",
      precision_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level Recall:",
      recall_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level F1:",
      f1_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level AUROC:",
      roc_auc_score(patient_y_true, patient_y_prob))

print("\nPatient-level confusion matrix:")
print(
    confusion_matrix(
        patient_y_true,
        patient_y_pred
    )
)

Patient-level Accuracy: 0.4444444444444444
Patient-level Precision: 0.6666666666666666
Patient-level Recall: 0.5714285714285714
Patient-level F1: 0.6153846153846154
Patient-level AUROC: 0.3571428571428571

Patient-level confusion matrix:
[[0 2]
 [3 4]]


In [47]:
print(patient_results)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          1.000000            1.000000           230   
1      chf03           1          0.699295            0.713641           240   
2      chf08           1          0.491855            0.508818           240   
3     chf206           1          0.616369            0.637882           251   
4     chf207           1          0.997221            0.997793           227   
5     chf209           1          0.381264            0.384402           226   
6     chf214           0          0.986767            0.987077           203   
7     chf217           0          0.639977            0.633862           279   
8     chf228           1          0.495592            0.507719           273   

   predicted_class  
0                1  
1                1  
2                0  
3                1  
4                1  
5                0  
6                1  
7                1  
8         

In [48]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully")

Random Forest trained successfully


In [49]:
rf_y_pred = rf_model.predict(X_test)

rf_y_prob = rf_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated")

Predictions generated


In [50]:
print("Random Forest — Window-level Results")
print("------------------------------------")

print("Accuracy :",
      accuracy_score(y_test, rf_y_pred))

print("Precision:",
      precision_score(y_test, rf_y_pred, zero_division=0))

print("Recall   :",
      recall_score(y_test, rf_y_pred, zero_division=0))

print("F1 Score :",
      f1_score(y_test, rf_y_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(y_test, rf_y_prob))

Random Forest — Window-level Results
------------------------------------
Accuracy : 0.764407561088059
Precision: 0.7747663551401869
Recall   : 0.982809721398933
F1 Score : 0.866475045727724
AUROC    : 0.08799226203799128


In [51]:
rf_test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

rf_test_results["predicted_probability"] = rf_y_prob

In [52]:
rf_patient_results = (
    rf_test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [53]:
rf_patient_results["predicted_class"] = (
    rf_patient_results["mean_probability"] >= 0.5
).astype(int)

In [54]:
print(rf_patient_results)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          0.986261            0.983333           230   
1      chf03           1          0.882000            0.876667           240   
2      chf08           1          0.670750            0.670000           240   
3     chf206           1          0.583519            0.606667           251   
4     chf207           1          0.984684            0.983333           227   
5     chf209           1          0.635413            0.636667           226   
6     chf214           0          0.994811            0.996667           203   
7     chf217           0          0.985448            0.986667           279   
8     chf228           1          0.861917            0.863333           273   

   predicted_class  
0                1  
1                1  
2                1  
3                1  
4                1  
5                1  
6                1  
7                1  
8         

In [55]:
rf_patient_y_true = rf_patient_results["risk_label"]
rf_patient_y_pred = rf_patient_results["predicted_class"]
rf_patient_y_prob = rf_patient_results["mean_probability"]

In [56]:
print("Random Forest — Patient-level Results")
print("-------------------------------------")

print("Accuracy :",
      accuracy_score(
          rf_patient_y_true,
          rf_patient_y_pred
      ))

print("Precision:",
      precision_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("Recall   :",
      recall_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("F1 Score :",
      f1_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("AUROC    :",
      roc_auc_score(
          rf_patient_y_true,
          rf_patient_y_prob
      ))

Random Forest — Patient-level Results
-------------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.07142857142857142


In [57]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        rf_patient_y_true,
        rf_patient_y_pred
    )
)


Confusion Matrix:
[[0 2]
 [0 7]]


In [58]:
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(rf_importance)

        feature  importance
10  pnn50_trend    0.325637
9   rmssd_trend    0.271522
8    sdnn_trend    0.236236
6         lf_hf    0.042864
0       mean_rr    0.030792
7      rr_count    0.022315
4      lf_power    0.021733
2         rmssd    0.015503
3         pnn50    0.013646
1          sdnn    0.010405
5      hf_power    0.009347


In [63]:
#Random forest without trends

In [64]:
baseline_features = [
    "mean_rr",
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_power",
    "hf_power",
    "lf_hf",
    "rr_count"
]

In [65]:
X_train_base = train_df[baseline_features]
X_test_base = test_df[baseline_features]

In [66]:
rf_base = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_base.fit(X_train_base, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [67]:
rf_base_prob = rf_base.predict_proba(X_test_base)[:, 1]
rf_base_pred = rf_base.predict(X_test_base)

In [68]:
rf_base_results = test_df[
    ["patient_id", "risk_label"]
].copy()

rf_base_results["predicted_probability"] = rf_base_prob

In [69]:
rf_base_patient = (
    rf_base_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [70]:
rf_base_patient["predicted_class"] = (
    rf_base_patient["mean_probability"] >= 0.5
).astype(int)

In [72]:
rf_base_true = rf_base_patient["risk_label"]
rf_base_pred = rf_base_patient["predicted_class"]
rf_base_prob = rf_base_patient["mean_probability"]

In [73]:
print("Random Forest — HRV Only")
print("------------------------")

print("Accuracy :",
      accuracy_score(rf_base_true, rf_base_pred))

print("Precision:",
      precision_score(rf_base_true, rf_base_pred, zero_division=0))

print("Recall   :",
      recall_score(rf_base_true, rf_base_pred, zero_division=0))

print("F1 Score :",
      f1_score(rf_base_true, rf_base_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(rf_base_true, rf_base_prob))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        rf_base_true,
        rf_base_pred
    )
)

Random Forest — HRV Only
------------------------
Accuracy : 0.6666666666666666
Precision: 0.75
Recall   : 0.8571428571428571
F1 Score : 0.8
AUROC    : 0.42857142857142855

Confusion Matrix:
[[0 2]
 [1 6]]


In [74]:
print(rf_base_patient)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          0.583261            0.693333           230   
1      chf03           1          0.805194            0.910000           240   
2      chf08           1          0.625028            0.636667           240   
3     chf206           1          0.582337            0.576667           251   
4     chf207           1          0.384258            0.300000           227   
5     chf209           1          0.839277            0.880000           226   
6     chf214           0          0.689442            0.746667           203   
7     chf217           0          0.620741            0.640000           279   
8     chf228           1          0.640781            0.666667           273   

   predicted_class  
0                1  
1                1  
2                1  
3                1  
4                0  
5                1  
6                1  
7                1  
8         

In [75]:
import xgboost

print("XGBoost version:", xgboost.__version__)

XGBoost version: 3.2.0


In [76]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost trained successfully")

XGBoost trained successfully


In [77]:
xgb_y_pred = xgb_model.predict(X_test)

xgb_y_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated")

Predictions generated


In [78]:
print("XGBoost — Window-level Results")
print("------------------------------")

print("Accuracy :",
      accuracy_score(y_test, xgb_y_pred))

print("Precision:",
      precision_score(y_test, xgb_y_pred, zero_division=0))

print("Recall   :",
      recall_score(y_test, xgb_y_pred, zero_division=0))

print("F1 Score :",
      f1_score(y_test, xgb_y_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(y_test, xgb_y_prob))

XGBoost — Window-level Results
------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.16999842584371086


In [79]:
xgb_test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

xgb_test_results["predicted_probability"] = xgb_y_prob

In [81]:
xgb_patient_results = (
    xgb_test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [82]:
xgb_patient_results["predicted_class"] = (
    xgb_patient_results["mean_probability"] >= 0.5
).astype(int)

In [83]:
xgb_patient_y_true = xgb_patient_results["risk_label"]
xgb_patient_y_pred = xgb_patient_results["predicted_class"]
xgb_patient_y_prob = xgb_patient_results["mean_probability"]

In [84]:
print("XGBoost — Patient-level Results")
print("--------------------------------")

print("Accuracy :",
      accuracy_score(
          xgb_patient_y_true,
          xgb_patient_y_pred
      ))

print("Precision:",
      precision_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("Recall   :",
      recall_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("F1 Score :",
      f1_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("AUROC    :",
      roc_auc_score(
          xgb_patient_y_true,
          xgb_patient_y_prob
      ))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        xgb_patient_y_true,
        xgb_patient_y_pred
    )
)

XGBoost — Patient-level Results
--------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.2857142857142857

Confusion Matrix:
[[0 2]
 [0 7]]


In [85]:
xgb_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(xgb_importance)

        feature  importance
10  pnn50_trend    0.320437
8    sdnn_trend    0.211731
6         lf_hf    0.191194
9   rmssd_trend    0.160247
0       mean_rr    0.041035
1          sdnn    0.028856
3         pnn50    0.016424
4      lf_power    0.010863
7      rr_count    0.010331
5      hf_power    0.005551
2         rmssd    0.003332


In [86]:
baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest - HRV Only",
        "Random Forest - HRV + Trends",
        "XGBoost - HRV + Trends"
    ],
    "Accuracy": [
        0.444444,
        0.666667,
        0.777778,
        0.777778
    ],
    "Precision": [
        0.666667,
        0.750000,
        0.777778,
        0.777778
    ],
    "Recall": [
        0.571429,
        0.857143,
        1.000000,
        1.000000
    ],
    "F1": [
        0.615385,
        0.800000,
        0.875000,
        0.875000
    ],
    "AUROC": [
        0.357143,
        0.428571,
        0.071429,
        0.285714
    ]
})

baseline_results

,Model,Accuracy,Precision,Recall,F1,AUROC
0,Logistic Regression,0.444444,0.666667,0.571429,0.615385,0.357143
1,Random Forest - HRV Only,0.666667,0.750000,0.857143,0.800000,0.428571
2,Random Forest - HRV + Trends,0.777778,0.777778,1.000000,0.875000,0.071429
3,XGBoost - HRV + Trends,0.777778,0.777778,1.000000,0.875000,0.285714


In [87]:
from sklearn.model_selection import GroupKFold

In [90]:
print("X shape:", X.shape)
print("y shape:", y.shape)

patient_ids = ml_dataset["patient_id"]

print("Patient groups:", len(patient_ids))
print("Unique patients:", patient_ids.nunique())

X shape: (11109, 11)
y shape: (11109,)
Patient groups: 11109
Unique patients: 44


In [91]:
groups = ml_dataset["patient_id"]

print("Total samples:", len(X))
print("Total patients:", groups.nunique())

Total samples: 11109
Total patients: 44


In [92]:
patient_label_check = (
    ml_dataset[
        ["patient_id", "risk_label"]
    ]
    .drop_duplicates()
)

print(patient_label_check.shape)

print(
    patient_label_check["risk_label"].value_counts()
)

print(
    patient_label_check.groupby("patient_id")["risk_label"].nunique().max()
)

(44, 2)
risk_label
1    32
0    12
Name: count, dtype: int64
1


In [93]:
group_kfold = GroupKFold(n_splits=5)

folds = list(
    group_kfold.split(
        X,
        y,
        groups=groups
    )
)

print("Number of folds:", len(folds))

Number of folds: 5


In [94]:
for fold, (train_idx, val_idx) in enumerate(folds, start=1):

    train_patients = set(
        groups.iloc[train_idx]
    )

    val_patients = set(
        groups.iloc[val_idx]
    )

    overlap = train_patients.intersection(
        val_patients
    )

    print(
        f"Fold {fold}: "
        f"Train patients = {len(train_patients)}, "
        f"Validation patients = {len(val_patients)}, "
        f"Overlap = {len(overlap)}"
    )

Fold 1: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 2: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 3: Train patients = 36, Validation patients = 8, Overlap = 0
Fold 4: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 5: Train patients = 35, Validation patients = 9, Overlap = 0


In [95]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [96]:
def evaluate_cv_model(model, X, y, groups, folds):

    fold_results = []

    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

        X_train_cv = X.iloc[train_idx]
        X_val_cv = X.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model.fit(X_train_cv, y_train_cv)

        y_pred_cv = model.predict(X_val_cv)
        y_prob_cv = model.predict_proba(X_val_cv)[:, 1]

        result = {
            "fold": fold_number,
            "accuracy": accuracy_score(y_val_cv, y_pred_cv),
            "precision": precision_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "recall": recall_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "f1": f1_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "auroc": roc_auc_score(
                y_val_cv,
                y_prob_cv
            )
        }

        fold_results.append(result)

    return pd.DataFrame(fold_results)

In [99]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [100]:
lr_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

In [101]:
lr_cv_results = evaluate_cv_model(
    lr_cv,
    X,
    y,
    groups,
    folds
)

lr_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.538156,1.000000,0.475188,0.644241,0.830635
1,2,0.787744,0.738750,0.951691,0.831809,0.868801
2,3,0.537415,1.000000,0.463057,0.632999,0.858901
3,4,0.563137,0.828859,0.558192,0.667117,0.599464
4,5,0.551648,0.536597,0.994068,0.696970,0.728045


In [102]:
print("Logistic Regression — GroupKFold Mean")
print("-------------------------------------")

print(
    lr_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

Logistic Regression — GroupKFold Mean
-------------------------------------
accuracy     0.595620
precision    0.820841
recall       0.688439
f1           0.694627
auroc        0.777169
dtype: float64


In [103]:
xgb_cv = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_cv_results = evaluate_cv_model(
    xgb_cv,
    X,
    y,
    groups,
    folds
)

xgb_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.905161,1.000000,0.892231,0.943046,1.000000
1,2,0.680284,0.785558,0.578100,0.666048,0.666339
2,3,0.630709,0.819962,0.732092,0.773540,0.441840
3,4,0.672574,0.858733,0.697175,0.769567,0.615602
4,5,0.518681,0.518681,1.000000,0.683068,0.584034


In [104]:
print("XGBoost — GroupKFold Mean")
print("-------------------------")

print(
    xgb_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

XGBoost — GroupKFold Mean
-------------------------
accuracy     0.681482
precision    0.796587
recall       0.779920
f1           0.767054
auroc        0.661563
dtype: float64


In [105]:
rf_cv = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_cv_results = evaluate_cv_model(
    rf_cv,
    X,
    y,
    groups,
    folds
)

rf_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.831495,0.877045,0.940351,0.907596,0.802969
1,2,0.791741,0.831760,0.780193,0.805152,0.629597
2,3,0.746356,0.843493,0.866328,0.854758,0.557000
3,4,0.659725,0.855824,0.680791,0.758339,0.674301
4,5,0.518681,0.518681,1.000000,0.683068,0.628254


In [106]:
print("Random Forest — GroupKFold Mean")
print("--------------------------------")

print(
    rf_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

Random Forest — GroupKFold Mean
--------------------------------
accuracy     0.709600
precision    0.785361
recall       0.853533
f1           0.801782
auroc        0.658424
dtype: float64
